In [1]:
import pandas as pd
import json
#from google.colab import drive
#drive.mount('/content/drive')


In [3]:
with open('data/layer1.json', 'r') as f:
    data1 = json.load(f)

In [4]:
import pandas as pd
import json
import os

def create_normalized_dataframes(recipes_data):
    """
    Crée trois dataframes normalisés:
    1. recipes: informations générales sur chaque recette
    2. recipe_ingredients: un ingrédient par ligne
    3. recipe_instructions: une instruction par ligne
    """

    #  recipes (informations générales)
    recipes_list = []
    for recipe in recipes_data:
         # Extraire les ingrédients et les instructions en une seule chaîne
        ingredients = [item['text'] for item in recipe['ingredients']]
        instructions = [item['text'] for item in recipe['instructions']]
        recipe_dict = {
            'id': recipe['id'],
            'title': recipe['title'],
            'url': recipe['url'],
            'partition': recipe.get('partition', ''),
            'number_of_ingredients': len(ingredients),
            'number_of_steps': len(instructions)
        }
        recipes_list.append(recipe_dict)
    recipes_df = pd.DataFrame(recipes_list)

    # recipe_ingredients (un ingrédient par ligne)
    ingredients_list = []
    for recipe in recipes_data:
        recipe_id = recipe['id']
        for ingredient in recipe['ingredients']:
            ingredients_list.append({
                'id': recipe_id,
                'ingredient': ingredient['text']
            })
    recipe_ingredients_df = pd.DataFrame(ingredients_list)

    #  recipe_instructions (une instruction par ligne)
    instructions_list = []
    for recipe in recipes_data:
        recipe_id = recipe['id']
        for i, instruction in enumerate(recipe['instructions']):
            instructions_list.append({
                'id': recipe_id,
                'step_number': i + 1,
                'instruction': instruction['text']
            })
    recipe_instructions_df = pd.DataFrame(instructions_list)
    recipe_instructions_df = recipe_instructions_df[~recipe_instructions_df['instruction'].str.startswith('(')]

    return recipes_df, recipe_ingredients_df, recipe_instructions_df

def save_dataframes(dataframes, output_dir='.'):
    """Sauvegarde les dataframes en CSV"""
    names = ['recipes', 'recipe_ingredients', 'recipe_instructions']

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for df, name in zip(dataframes, names):
        output_path = os.path.join(output_dir, f"{name}.csv")
        df.to_csv(output_path, index=False)
        print(f"Dataframe {name} sauvegardé dans {output_path}")

# Exemple d'utilisation
if __name__ == "__main__":

    # Créer les trois dataframes normalisés
    recipes_df, recipe_ingredients_df, recipe_instructions_df = create_normalized_dataframes(data1)

    print(f"Nombre total de recettes: {len(recipes_df)}")


    print(f"Nombre total d'ingrédients: {len(recipe_ingredients_df)}")


    print(f"Nombre total d'instructions: {len(recipe_instructions_df)}")

    # Sauvegarder les dataframes
    save_dataframes([recipes_df, recipe_ingredients_df, recipe_instructions_df])



Nombre total de recettes: 1029720
Nombre total d'ingrédients: 9605394
Nombre total d'instructions: 10666639
Dataframe recipes sauvegardé dans .\recipes.csv
Dataframe recipe_ingredients sauvegardé dans .\recipe_ingredients.csv
Dataframe recipe_instructions sauvegardé dans .\recipe_instructions.csv


In [ ]:
import os
def segmentation_dataset(df, max_size_segment ,output_dir, nameoffile):

    """
    cette fonction permet de segmenter mes instructions par dataset de 100000 instruction max afin de faciliter le processus de traitement
    ceci en se rassurant que les instruction d"une recette se retrouve uniquement dans un dataset
    """
    # D'abord grouper complètement par ID
    id_counts = df.groupby('id').size().reset_index(name='count')
   
    # Puis assigner les segments
    id_counts['cumulative_count'] = id_counts['count'].cumsum()
    id_counts['id_ter'] = (id_counts['cumulative_count'] - 1) //  max_size_segment + 1
    
    # Fusionner les segments avec le dataset
    df_with_segments =  pd.merge(df, id_counts, on= 'id', how= 'inner')[['id','step_number','instruction','id_ter']]

    
    
    os.makedirs(output_dir, exist_ok=True)

    # Grouper par id_sec et sauvegarder
    for segment_id, group in df_with_segments.groupby('id_ter'):
        filename = f"{output_dir}/{nameoffile}_{segment_id}.csv"
        group.to_csv(filename, index=False)
        print(f"Segment {segment_id}: {len(group)} lignes, {group['id'].nunique()} IDs uniques")

segmentation_dataset(sample
                     ,10000,"subset_instructions_segment", "subset")

In [6]:
recipes_df.head(16)

,id,title,url,partition,number_of_ingredients,number_of_steps
0,000018c8a5,Worlds Best Mac and Cheese,http://www.epicurious.com/recipes/food/views/-...,train,14,23
1,000033e39b,Dilly Macaroni Salad Recipe,http://cookeatshare.com/recipes/dilly-macaroni...,train,9,8
2,000035f7ed,Gazpacho,http://www.foodnetwork.com/recipes/gazpacho1.html,train,9,4
3,00003a70b1,Crunchy Onion Potato Bake,http://www.food.com/recipe/crunchy-onion-potat...,test,7,9
4,00004320bb,Cool 'n Easy Creamy Watermelon Pie,http://www.food.com/recipe/cool-n-easy-creamy-...,train,5,7
5,0000631d90,Easy Tropical Beef Skillet,http://www.food.com/recipe/easy-tropical-beef-...,train,10,6
6,000075604a,Kombu Tea Grilled Chicken Thigh,https://cookpad.com/us/recipes/150100-kombu-te...,train,3,5
7,00007bfd16,Strawberry Rhubarb Dump Cake,http://www.food.com/recipe/strawberry-rhubarb-...,train,7,2
8,000095fc1d,Yogurt Parfaits,http://tastykitchen.com/recipes/breakfastbrunc...,train,3,1
9,0000973574,Zucchini Nut Bread,http://www.food.com/recipe/zucchini-nut-bread-...,train,11,7


In [7]:
recipe_ingredients = pd.read_csv("recipe_ingredients.csv")
recipe_ingredients.head(16)

,id,ingredient
0,000018c8a5,6 ounces penne
1,000018c8a5,2 cups Beechers Flagship Cheese Sauce (recipe ...
2,000018c8a5,"1 ounce Cheddar, grated (1/4 cup)"
3,000018c8a5,"1 ounce Gruyere cheese, grated (1/4 cup)"
4,000018c8a5,1/4 to 1/2 teaspoon chipotle chili powder (see...
5,000018c8a5,1/4 cup (1/2 stick) unsalted butter
6,000018c8a5,1/3 cup all-purpose flour
7,000018c8a5,3 cups milk
8,000018c8a5,"14 ounces semihard cheese (page 23), grated (a..."
9,000018c8a5,"2 ounces semisoft cheese (page 23), grated (1/..."


In [8]:
cooking_verbs = [
    'aerate', 'age', 'arrange', 'bake', 'barbecue', 'baste', 'batter', 'beat', 'bind', 'blacken', 'blanch',
    'blend', 'boil', 'bone', 'braise', 'bread', 'brew', 'broil', 'brown', 'brush', 'burn', 'butterfly',
    'can', 'caramelize', 'char', 'char-broil', 'chill', 'chop', 'chunk', 'churn', 'clarify', 'coddle',
    'combine', 'congeal', 'cool', 'core', 'cream', 'cured', 'cut', 'debone', 'decorate', 'deep fry',
    'deglaze', 'descale', 'devil', 'dice', 'dip', 'dough', 'drain', 'drizzle', 'dry', 'escallop',
    'ferment', 'fillet', 'filter', 'flambe', 'flavor', 'flip', 'fold', 'freeze', 'french fry',
    'fricassee', 'frost', 'froth', 'fry', 'garnish', 'gel', 'glaze', 'grate', 'gratin', 'grease',
    'grill', 'grind', 'hard boil', 'harden', 'hash', 'heat', 'hull', 'ice', 'infuse', 'insert',
    'jell', 'juice', 'julienne', 'knead', 'layer', 'lay', 'leaven', 'leave', 'lift', 'macerate',
    'make', 'marinate', 'mash', 'measure', 'melt', 'microwave', 'mince', 'mix', 'mold', 'moisten',
    'mound', 'oil', 'open', 'oven fry', 'overcook', 'pack', 'paint', 'pan fry', 'parboil', 'pare',
    'peel', 'percolate', 'pickle', 'pierce', 'pit', 'poach', 'pop', 'pour', 'precook', 'prepare',
    'preserve', 'press', 'pressure-cook', 'prick', 'process', 'pulp', 'puree', 'push', 'quarter',
    'raw', 'raise',  'reduce', 'refresh', 'refrigerate', 'reheat', 'render', 'replace',
    'return', 'rise', 'ring', 'roast', 'roll', 'rub', 'salt', 'saute', 'scald', 'scallop', 'scatter',
    'scoop', 'score', 'scramble', 'scrape', 'scrub', 'sear', 'season', 'separate', 'serve', 'set',
    'settle', 'shave', 'shell', 'shirr', 'shred', 'shuck', 'sieve', 'sift', 'simmer', 'skewer',
    'skim', 'skin', 'slice', 'slid', 'slide', 'slip', 'slit', 'sliver', 'slow cook', 'smear', 'smoke',
    'snip', 'soak', 'soft boil', 'souse', 'spoon', 'spread', 'sprinkle', 'steam', 'steep', 'stew',
    'stir', 'stir fry', 'strain', 'strew', 'stuff', 'surround', 'sweeten', 'taste', 'temper',
    'tenderize', 'thicken', 'thin', 'tie', 'tilt', 'tip', 'toast', 'top', 'toss', 'trim', 'truss',
    'turn', 'twist', 'uncured', 'unmold', 'warm', 'wash', 'wedge', 'whip', 'whisk', 'wind', 'wrap',
    'zap', 'zest'
]
cooking_verbs = [
    'aerate', 'age', 'air fry', 'arrange', 'bake', 'barbecue', 'baste', 'batter', 'beat', 'bind', 'blacken',
    'blanch', 'blend', 'boil', 'bone', 'braise', 'bread', 'brew', 'broil', 'brown', 'brush', 'burn', 'butterfly',
    'can', 'caramelize', 'carve', 'char', 'char-broil', 'chill', 'chop', 'chunk', 'churn', 'clarify', 'coddle',
    'coat', 'combine', 'confit', 'congeal', 'cool', 'core', 'cream', 'cure', 'cured', 'cut', 'debone',
    'decorate', 'deep fry', 'deglaze', 'dehydrate', 'descale', 'devil', 'devein', 'dice', 'dip', 'dissolve',
    'dot', 'dough', 'drain', 'drizzle', 'dry', 'dust', 'emulsify', 'enrich', 'escallop', 'extract', 'ferment',
    'fillet', 'filter', 'fizz', 'flambe', 'flavor', 'flip', 'foam', 'fold', 'french fry', 'freeze', 'fricassee',
    'frost', 'froth', 'fry', 'garnish', 'gel', 'glaze', 'grate', 'gratin', 'grease', 'grill', 'grind',
    'hard boil', 'harden', 'hash', 'heat', 'hull', 'hydrate', 'ice', 'infuse', 'insert', 'jell', 'juice',
    'julienne', 'knead', 'layer', 'lay', 'leaven', 'leave', 'lift', 'macerate', 'make', 'marinate', 'mash',
    'measure', 'melt', 'microwave', 'mill', 'mince', 'mix', 'mold', 'moisten', 'mound', 'muddle', 'oil', 'open',
    'oven fry', 'overcook', 'pack', 'paint', 'pan fry', 'parboil', 'pare', 'parch', 'peel', 'percolate',
    'pickle', 'pierce', 'pipe', 'pit', 'plate', 'poach', 'pop', 'popcorn', 'pour', 'precook', 'prepare',
    'preserve', 'press', 'pressurize', 'pressure-cook', 'prick', 'process', 'pulp', 'pull', 'puree', 'push',
    'quarter', 'raw', 'raise', 'reconstitute', 'reduce', 'refresh', 'refrigerate', 'reheat', 'render', 'replace',
    'resin', 'return', 'rim', 'rise', 'ring', 'roast', 'roll', 'rub', 'salt', 'saute', 'scald', 'scallop',
    'scatter', 'scoop', 'score', 'scramble', 'scrape', 'scrub', 'sear', 'season', 'separate', 'serve', 'set',
    'settle', 'shave', 'shell', 'shirr', 'shred', 'shuck', 'sieve', 'sift', 'simmer', 'skewer', 'skim', 'skin',
    'slice', 'slide', 'slip', 'slit', 'slather', 'sliver', 'slow cook', 'smear', 'smoke', 'smother', 'snap',
    'snip', 'soak', 'soft boil', 'souse', 'splash', 'spike', 'spoon', 'spread', 'sprinkle', 'spurt', 'squeeze',
    'steam', 'steep', 'stew', 'stir', 'stir fry', 'strain', 'stretch', 'strew', 'stuff', 'surround', 'sweeten',
    'taste', 'temper', 'tenderize', 'thicken', 'thin', 'tie', 'tilt', 'tip', 'toast', 'top', 'torch', 'toss',
    'trim', 'truss', 'turn', 'twist', 'uncured', 'unmold', 'vacuum seal', 'warm', 'wash', 'wedge', 'whip',
    'whisk', 'wind', 'wrap', 'zap', 'zest'
]